# Partitioning Fundamentals: Parallelism and Data Distribution

**Module**: 05_partitions_and_shuffle  
**Notebook**: 01_partitioning_fundamentals.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark partitions datasets, the correlation between partitions and executor tasks, and inspecting row distributions.
- **Why does this topic exist?**: Partitions dictate cluster concurrency, memory footprint, and network transfer volume.
- **What problem does it solve?**: Improper partition sizing causes either underutilized hardware or out-of-memory errors.
- **Where is it used in real-world Data Engineering?**: Sizing data lake ingestion and tuning shuffle partitions.

---

## 2. Partition Sizing Heuristics

- **Target Size**: 100 MB to 200 MB in memory per partition.
- **Cluster Ratio**: 2x to 4x total executor cores in the cluster.
- Our Docker Cluster: 2 workers x 2 cores = 4 cores -> Target 8 to 16 partitions for active processing.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Partitions_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Inspecting File Partitions on Retail Orders (60,000 rows)

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
print(f"Total orders: {orders.count()}")
print(f"Initial partition count from disk: {orders.rdd.getNumPartitions()}")

# Inspect row distribution per partition using glom
dist = orders.rdd.glom().map(len).collect()
print(f"Rows per partition: {dist}")

---

## Experiment: Parallelism Scaling with Partition Count

- **1. What are we testing?**: Compare execution time of heavy CPU map across 1, 2, and 4 partitions.
- **2. What do we expect?**: Execution time scales inversely with partition count up to 4 cores.

In [ ]:
def cpu_heavy(row):
    return sum(i*i for i in range(1000))

for p in [1, 2, 4]:
    t0 = time.time()
    orders.repartition(p).rdd.map(cpu_heavy).count()
    print(f"Partitions {p}: {time.time()-t0:.3f}s")

**3. What happened?**: Increasing partitions up to total cores decreased runtime.  
**4. Why did it happen?**: More cores were actively utilized in parallel.  
**5. Practical Engineering Lesson**: Always maintain at least 1 partition per available core.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Default Shuffle Partitions
**Problem Statement**: Retrieve spark.sql.shuffle.partitions.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Custom Repartition and Glom
**Problem Statement**: Repartition orders into 6 partitions and print min/max partition sizes.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Partition Count Calculation
**Problem Statement**: Write a Python function to compute optimal partition count given dataset size in MB and core count.  
**Input / Context**: Formula

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Default Shuffle Partitions):
```python
print(spark.conf.get('spark.sql.shuffle.partitions'))
```

#### Solution 2 (Custom Repartition and Glom):
```python
s = orders.repartition(6).rdd.glom().map(len).collect()
print('Sizes:', s,'Min:', min(s),'Max:', max(s))
```

#### Solution 3 (Partition Count Calculation):
```python
def calc_partitions(size_mb, cores): return max(cores * 2, int(size_mb / 128))
print('Optimal for 10GB on 16 cores:', calc_partitions(10240, 16))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Sizing Rule**: 100-200 MB per partition; 2-4x total cores.

### Essential Technical Terms
- `Partition`: Atomic unit of distributed data.
- `glom`: Inspects partition contents.

### Common Engineering Mistakes
- **Single partition**: Serializes execution on 1 core.

### Interview Questions on This Topic
- **Q: How does partition count affect task count?**
  - *A*: 1:1 correlation per stage.

### What to Study Next
Open 02_repartition_and_coalesce.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')